In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Put your file path here
train_file_path = '/content/drive/MyDrive/Datasets/creditscore/cs-training.csv'
test_file_path = '/content/drive/MyDrive/Datasets/creditscore/cs-test.csv'
import pandas as pd
train = pd.read_csv(train_file_path)
train.head()
test = pd.read_csv(test_file_path)
test.head()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_hist_gradient_boosting  # Needed for older versions
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt


# Separate target and features
X_train = train.drop(columns=['SeriousDlqin2yrs'])
y_train = train['SeriousDlqin2yrs']

X_test = test.drop(columns=['SeriousDlqin2yrs'])  # All NaN target
y_test = None  # No labels

# 1. Preprocessing
numeric_features = X_train.columns.tolist()
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features)
])

# 2. Build the pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(random_state=42))
])

# 3. Hyperparameter tuning
param_grid = {
    'classifier__max_iter': [100, 200],
    'classifier__learning_rate': [0.05, 0.1],
    'classifier__max_depth': [3, 5],
    'classifier__min_samples_leaf': [20, 50]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation AUC:", grid_search.best_score_)

# 4. Final model
final_model = grid_search.best_estimator_

# 5. Predict probabilities on test set
y_proba = final_model.predict_proba(X_test)[:, 1]

# Plot probability distribution
plt.hist(y_proba, bins=50, color='skyblue')
plt.title('Predicted Probability Distribution')
plt.xlabel('Probability of Default')
plt.ylabel('Number of Applicants')
plt.show()

# 6. Threshold tuning
threshold = 0.2
y_pred_flagged = (y_proba >= threshold).astype(int)
print(f"Proportion flagged as risky: {y_pred_flagged.mean():.2f}")

# Optional: Save predictions
pd.DataFrame({
    'Probability': y_proba,
    'FlaggedAsRisky': y_pred_flagged
}).to_csv('predictions.csv', index=False)


In [ ]:
# Simulate predicted probabilities based on observed distribution pattern
np.random.seed(42)
predicted_probs = np.random.beta(a=0.5, b=5, size=100000)  # Simulate 100,000 applicants

# Thresholds to visualize
thresholds = [0.15, 0.2, 0.25]
proportions = []

# Calculate proportion flagged as risky for each threshold
for thresh in thresholds:
    risky = (predicted_probs >= thresh).mean()
    proportions.append(risky)

# Plot
plt.figure(figsize=(10, 6))
plt.hist(predicted_probs, bins=50, alpha=0.6, color='skyblue', edgecolor='black')
for thresh in thresholds:
    plt.axvline(thresh, linestyle='--', label=f'Threshold = {thresh}', linewidth=2)

plt.xlabel('Predicted Probability of Default')
plt.ylabel('Number of Applicants')
plt.title('Effect of Threshold Adjustment on Predicted Probabilities')
plt.legend()
plt.grid(True)
plt.show()

# Print proportion flagged for each threshold
proportion_results = {f'Threshold {thresh}': f'{round(prop * 100, 2)}%' for thresh, prop in zip(thresholds, proportions)}
proportion_results
